# 📊 V-ChartText: Ứng dụng VLM trong Tự động Trích xuất & Sinh mô tả Biểu đồ Học tập Tiếng Việt
**Repository:** [dangthihoangnganit6/V-ChartText](https://github.com/dangthihoangnganit6/V-ChartText)  
**Mục tiêu Notebook:**
- **Bước 1 (Đọc ảnh & Tách số liệu):** Sử dụng VLM (`Qwen/Qwen2-VL-2B-Instruct`) trích xuất bảng số liệu chuẩn 7 trường: `chart_title`, `chart_type`, `unit`, `x_axis_label`, `categories`, `series_name`, `values`.
- **Bước 2 (Kiểm chứng số liệu - Validation Rule-based):** Xây dựng khung kiểm tra tự động (so khớp categories vs values, kiểm tra logic cơ cấu %, chuẩn hóa 7 loại biểu đồ).
- **Xử lý ngữ liệu tiếng Việt đặc thù:** Bảo toàn số thực có dấu phẩy (`12,5`), hàng nghìn dấu chấm (`1.200`), đơn vị ghép (`nghìn ha`, `triệu tấn`), biểu đồ nhiều chuỗi (` | ` và ` || `).
- **Chống crash Colab:** Parse JSON an toàn bằng regex + try/except, giải phóng VRAM bằng `torch.cuda.empty_cache()`, lưu kết quả cuốn chiếu (append CSV) kèm cơ chế phục hồi (resume checkpoint).

--- 
## ⚙️ Phần 1: Cài đặt Môi trường & Thư viện Cần thiết

In [ ]:
# Cài đặt các thư viện phục vụ mô hình VLM Qwen2-VL và xử lý dữ liệu
!pip install -q --upgrade pip
!pip install -q transformers==4.45.2 accelerate torchvision qwen-vl-utils pandas tqdm Pillow

print("✅ Cài đặt thư viện hoàn tất!")

--- 
## 📂 Phần 2: Mount Google Drive & Thiết lập Đường dẫn Dữ liệu

In [ ]:
import os
import sys
from google.colab import drive

# 1. Kết nối với Google Drive
print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')

# 2. Cấu hình đường dẫn thư mục ảnh và file kết quả CSV
# Ghi chú: Nhóm có thể thay đổi đường dẫn này nếu thư mục ảnh trên Drive nằm ở vị trí khác
IMAGE_DIR = "/content/drive/MyDrive/Ảnh biểu đồ"
OUTPUT_CSV = "/content/drive/MyDrive/vcharttext_extracted_results.csv"
LOCAL_CSV_BACKUP = "/content/vcharttext_extracted_results.csv"

print(f"\n📁 Thư mục ảnh nguồn: {IMAGE_DIR}")
print(f"💾 File kết quả CSV (Drive): {OUTPUT_CSV}")
print(f"💾 File kết quả dự phòng (Local): {LOCAL_CSV_BACKUP}")

if os.path.exists(IMAGE_DIR):
    print("✅ Đã tìm thấy thư mục ảnh trên Drive!")
else:
    print(f"⚠️ Lưu ý: Chưa tìm thấy '{IMAGE_DIR}'. Hãy kiểm tra lại tên thư mục trên Drive của bạn.")

--- 
## 🤖 Phần 3: Nạp Mô hình Qwen2-VL-2B-Instruct & Kiểm tra GPU

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# Kiểm tra GPU được cấp phát trên Colab (T4 / A100 / L4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị phần cứng: {device.upper()}")
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Card đồ họa: {gpu_name} ({vram_gb:.2f} GB VRAM)")

# Trỏ đến checkpoint chính thức của Qwen2-VL
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"\nĐang nạp processor và weights từ: {MODEL_ID}...")

# Cấu hình kích thước ảnh đọc vào tối ưu cho GPU T4 (tránh OOM mà vẫn bảo đảm độ nét của nhãn số)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ Mô hình {MODEL_ID} đã nạp thành công và sẵn sàng suy luận!")

--- 
## 📝 Phần 4: Thiết kế PROMPT_JSON Chuyên biệt & Các Hàm Bóc tách An toàn

### Bảng chuẩn 7 trường Ground-Truth (bám sát Slide 3, 4, 7, 9):
1. `chart_title`: Tiêu đề biểu đồ đầy đủ, nguyên văn tiếng Việt.
2. `chart_type`: Đúng 1 trong 7 nhãn: `column`, `cột ghép`, `cột chồng`, `đường`, `miền`, `tròn`, `kết hợp`.
3. `unit`: Đơn vị đo lường (ví dụ: `%`, `nghìn ha`, `triệu tấn`, `USD/người`, hoặc kết hợp `% và nghìn tấn`).
4. `x_axis_label`: Nhãn trục hoành hoặc tiêu chí đối tượng (ví dụ: `Năm`, `Vùng kinh tế`, `Quốc gia`, `Nhóm ngành`).
5. `categories`: Danh mục / các mốc thời gian cách nhau bằng ` | ` (ví dụ: `2010 | 2015 | 2020 | 2022`).
6. `series_name`: Tên các chuỗi dữ liệu (chú giải). Nếu đơn chuỗi ghi tên chuỗi; nếu nhiều chuỗi cách nhau bằng ` || `.
7. `values`: Chuỗi các giá trị số tương ứng. Cùng chuỗi cách nhau ` | `, giữa các chuỗi cách nhau ` || `.

**Quy tắc đặc thù ngữ liệu:**
- **Bảo toàn số thô:** Số thập phân dùng dấu phẩy (`12,5`), hàng nghìn dùng dấu chấm (`1.200`). KHÔNG tự đổi dấu chấm/phẩy bừa bãi.
- **Đối chiếu trục Y:** Nhìn thang chia vạch trên trục tung làm mốc đối chiếu trước khi đọc nhãn số để không phán đoán sai lệch.

In [ ]:
import re
import json
from PIL import Image
from qwen_vl_utils import process_vision_info

# PROMPT_JSON CHUYÊN BIỆT CHO BIỂU ĐỒ HỌC TẬP TIẾNG VIỆT
PROMPT_JSON = """Bạn là chuyên gia trích xuất dữ liệu biểu đồ giáo dục tiếng Việt (SGK Địa lí, Lịch sử, tài liệu ôn thi THPT Quốc gia).
Nhiệm vụ của bạn là đọc hình ảnh biểu đồ và trích xuất dữ liệu thành DUY NHẤT một chuỗi JSON hợp lệ theo đúng cấu trúc 7 trường sau:
{
  "chart_title": "Tên đầy đủ của biểu đồ (nguyên văn tiếng Việt có dấu, nếu không có tên ghi 'Không có tên')",
  "chart_type": "Loại biểu đồ, CHỈ ĐƯỢC CHỌN 1 TRONG 7 LOẠI: column | cột ghép | cột chồng | đường | miền | tròn | kết hợp",
  "unit": "Đơn vị đo lường (ví dụ: %, nghìn ha, triệu tấn, người, USD/người, tỉ đồng,... Nếu có 2 trục thì ghi cả hai ví dụ: % và nghìn tấn)",
  "x_axis_label": "Tên trục hoành hoặc tiêu chí phân loại (ví dụ: Năm, Vùng kinh tế, Quốc gia, Nhóm ngành,... Nếu không có ghi 'Không có')",
  "categories": "Danh sách các mốc/đối tượng trục hoành, phân tách nhau bằng ' | ' (ví dụ: 2010 | 2015 | 2020 | 2022)",
  "series_name": "Tên các chuỗi dữ liệu trong chú giải. Nếu đơn chuỗi ghi tên chuỗi đó; nếu nhiều chuỗi phân tách nhau bằng ' || ' (ví dụ: Trồng trọt || Chăn nuôi)",
  "values": "Các giá trị số đọc được tương ứng với categories. Các số trong cùng 1 chuỗi phân tách bằng ' | ', giữa các chuỗi phân tách bằng ' || '"
}

QUY TẮC BẮT BUỘC ĐỂ BẢO ĐẢM ĐỘ CHÍNH XÁC SỐ LIỆU:
1. ĐỐI CHIẾU THANG ĐO TRỤC TUNG (Y-axis):
   - BẮT BUỘC nhìn thang đo và vạch chia trên trục tung (ví dụ: 0, 500, 1000, 1500... hoặc 0, 10, 20...) làm mốc đối chiếu trước khi đọc nhãn số trên đỉnh cột/đường.
   - Dựa vào độ cao của cột hoặc vị trí điểm để xác nhận giá trị, tuyệt đối không đoán mò số liệu nằm ngoài thang đo.
2. BẢO TOÀN NGUYÊN BẢN KÝ TỰ SỐ TIẾNG VIỆT (KHÔNG ĐƯỢC TỰ Ý ĐỔI DẤU):
   - Biểu đồ tiếng Việt dùng dấu phẩy ',' cho số thập phân (ví dụ: 12,5) và dấu chấm '.' cho hàng nghìn (ví dụ: 1.200).
   - GIỮ NGUYÊN 100% dạng số nhìn thấy trên hình: ghi '12,5' thì giữ '12,5'; ghi '1.200' thì giữ '1.200'; ghi '6 586,6' thì giữ '6 586,6'.
   - TUYỆT ĐỐI KHÔNG tự tiện đổi dấu chấm thành phẩy hoặc phẩy thành chấm (không đổi '1.200' thành '1.2' nếu cột đang ở mốc trên 1000).
3. BIỂU ĐỒ NHIỀU CHUỖI (Cột ghép, cột chồng, đường, miền, kết hợp):
   - Trường 'categories': Ghi tất cả các mốc (ví dụ các năm: 2010 | 2015 | 2020 | 2022).
   - Trường 'series_name': Ghi tên từng chuỗi cách nhau bằng ' || ' (ví dụ: 'Nông nghiệp || Công nghiệp || Dịch vụ').
   - Trường 'values': Liệt kê giá trị tương ứng từng chuỗi. Các số trong 1 chuỗi cách nhau bằng ' | ', giữa các chuỗi cách nhau bằng ' || '.
   - Ví dụ với 2 chuỗi x 3 mốc năm: '12,5 | 15,0 | 18,2 || 5,4 | 6,8 | 8,1'
4. BIỂU ĐỒ TRÒN & CƠ CẤU:
   - Biểu đồ tròn đơn chuỗi: 'series_name' ghi tên đối tượng cơ cấu, 'values' ghi giá trị các phần cách nhau bằng ' | '.
   - Thường tổng các phần bằng 100%. Nếu các số ghi dạng '45,2' và '54,8' thì dấu phẩy là phần thập phân.
5. KHÔNG DỊCH TIẾNG VIỆT:
   - Giữ nguyên tiếng Việt có dấu, KHÔNG dịch tiêu đề, danh mục hay chú giải sang tiếng Anh.
6. ĐỊNH DẠNG ĐẦU RA:
   - Chỉ trả về DUY NHẤT một chuỗi JSON hợp lệ (bắt đầu bằng { và kết thúc bằng }), không kèm bất kỳ giải thích nào khác ngoài JSON.
"""

def safe_parse_json(raw_text):
    """
    Bóc tách JSON an toàn bằng regex và xử lý các lỗi cú pháp thường gặp của LLM.
    Trả về: (dict_data, error_message)
    """
    if not raw_text or not raw_text.strip():
        return None, "Output rỗng từ mô hình"

    # 1. Tìm khối JSON trong chuỗi trả về bằng regex
    match = re.search(r"```(?:json)?\s*(\{[\s\S]*?\})\s*```", raw_text)
    if match:
        json_str = match.group(1).strip()
    else:
        match_brace = re.search(r"(\{[\s\S]*\})", raw_text)
        if match_brace:
            json_str = match_brace.group(1).strip()
        else:
            return None, f"Không tìm thấy khối JSON hợp lệ: {raw_text[:120]}..."

    # 2. Xóa các dấu phẩy thừa trước dấu đóng ngoặc (trailing commas)
    json_str_clean = re.sub(r",\s*\}", "}", json_str)
    json_str_clean = re.sub(r",\s*\]", "]", json_str_clean)

    # 3. Thử parse JSON
    try:
        data = json.loads(json_str_clean)
        return data, None
    except Exception as e:
        # Fallback: thay dấu nháy đơn thành nháy kép
        try:
            fixed_quotes = json_str_clean.replace("'", '"')
            data = json.loads(fixed_quotes)
            return data, None
        except Exception:
            return None, f"Lỗi cú pháp JSON ({str(e)})"

def format_categories_field(val):
    """Chuẩn hóa trường categories thành chuỗi phân tách bởi ' | '"""
    if isinstance(val, list):
        return " | ".join([str(x).strip() for x in val])
    return str(val).strip() if val is not None else ""

def format_series_name_field(val):
    """Chuẩn hóa trường series_name thành chuỗi phân tách bởi ' || '"""
    if isinstance(val, list):
        return " || ".join([str(x).strip() for x in val])
    return str(val).strip() if val is not None else ""

def format_values_field(val):
    """Chuẩn hóa trường values: cùng chuỗi cách nhau ' | ', giữa các chuỗi cách nhau ' || '"""
    if isinstance(val, list):
        if len(val) > 0 and isinstance(val[0], list):
            return " || ".join([" | ".join([str(x).strip() for x in sub]) for sub in val])
        else:
            return " | ".join([str(x).strip() for x in val])
    return str(val).strip() if val is not None else ""

def extract_chart_data(image_path):
    """
    Truyền ảnh vào Qwen2-VL-2B-Instruct và trả về kết quả JSON trích xuất.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": PROMPT_JSON}
            ]
        }
    ]

    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False  # Dùng greedy decoding để đảm bảo tính xác thực tối đa của số liệu
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    raw_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # Dọn dẹp tensor trung gian
    del inputs, generated_ids, generated_ids_trimmed

    parsed_json, error_msg = safe_parse_json(raw_text)
    return parsed_json, raw_text, error_msg

print("✅ Thiết lập PROMPT_JSON và các hàm bóc tách dữ liệu hoàn tất!")

--- 
## 🔍 Phần 5: Khung Kiểm chứng Số liệu (Bước 2: Rule-Based Validation)

Theo **Kiến trúc 4 bước (Slide 6 & Slide 9)**:
- **Bước 2 (Kiểm chứng bảng):** Áp dụng hệ luật hợp lệ để đối chiếu dữ liệu trích xuất trước khi chuyển sang Bước 3 (Sinh mô tả).
- **Hệ thống luật kiểm chứng tự động:**
  1. **Khớp số lượng đối tượng:** Số danh mục trong `categories` ($N_{cat}$) phải khớp với số lượng phần tử trong từng chuỗi giá trị của `values`.
  2. **Khớp số lượng chuỗi:** Số lượng chuỗi trong `series_name` ($N_{series}$) phải khớp với số chuỗi dữ liệu trong `values` (phân tách bởi `||`).
  3. **Chuẩn hóa nhãn biểu đồ:** Kiểm tra xem `chart_type` có thuộc danh sách 7 loại chuẩn (`column`, `cột ghép`, `cột chồng`, `đường`, `miền`, `tròn`, `kết hợp`) hay không.
  4. **Kiểm tra cơ cấu 100%:** Với biểu đồ tròn hoặc miền có đơn vị `%`, tính tổng cơ cấu để kiểm tra xem có xấp xỉ 100% hay không (ngưỡng chấp nhận 95% - 105% do sai số làm tròn).
  5. **Phân loại trạng thái:** Gán nhãn `Hợp lệ (Pass)`, `Cần kiểm tra (Warning)`, hoặc `Lỗi trích xuất (Error)`.

In [ ]:
VALID_CHART_TYPES = ["column", "cột ghép", "cột chồng", "đường", "miền", "tròn", "kết hợp"]

def validate_extracted_data(data):
    """
    Khung kiểm chứng bảng số liệu (Bước 2 trong Kiến trúc 4 bước).
    Trả về: (status, notes_str, cleaned_dict)
      - status: 'Hợp lệ' | 'Cần kiểm tra' | 'Lỗi trích xuất'
      - notes_str: chi tiết lý do cảnh báo hoặc xác nhận hợp lệ
      - cleaned_dict: dữ liệu 7 trường đã chuẩn hóa sạch
    """
    if not data or not isinstance(data, dict):
        return "Lỗi trích xuất", "Không có dữ liệu JSON hợp lệ", {}

    notes = []
    status = "Hợp lệ"

    # 1. Trích xuất và chuẩn hóa 7 trường dữ liệu chuẩn
    chart_title = str(data.get("chart_title", "")).strip()
    raw_chart_type = str(data.get("chart_type", "")).strip().lower()
    unit = str(data.get("unit", "")).strip()
    x_axis_label = str(data.get("x_axis_label", "")).strip()
    categories_str = format_categories_field(data.get("categories"))
    series_name_str = format_series_name_field(data.get("series_name"))
    values_str = format_values_field(data.get("values"))

    # 2. Chuẩn hóa chart_type theo 7 loại chuẩn
    chart_type = raw_chart_type
    matched_type = None
    for ct in VALID_CHART_TYPES:
        if ct in raw_chart_type:
            matched_type = ct
            break
    if matched_type:
        chart_type = matched_type
    else:
        notes.append(f"Loại biểu đồ '{raw_chart_type}' chưa khớp 7 loại chuẩn")
        status = "Cần kiểm tra"

    # 3. Kiểm tra tính đầy đủ của trường cơ bản
    if not chart_title or chart_title.lower() in ["không có tên", "không có", "tiêu đề"]:
        notes.append("Thiếu tiêu đề biểu đồ rõ ràng")

    # 4. Phân tách danh mục và chuỗi giá trị để kiểm chứng ma trận
    cat_list = [c.strip() for c in categories_str.split("|") if c.strip()]
    series_list = [s.strip() for s in series_name_str.split("||") if s.strip()]
    val_series_list = [v.strip() for v in values_str.split("||") if v.strip()]

    num_cats = len(cat_list)
    num_series = len(series_list)
    num_val_series = len(val_series_list)

    if num_cats == 0:
        notes.append("Thiếu categories")
        status = "Cần kiểm tra"
    if num_val_series == 0:
        notes.append("Thiếu values")
        status = "Cần kiểm tra"

    # Đối chiếu số lượng chuỗi (series_name vs values)
    if num_series > 0 and num_val_series > 0 and num_series != num_val_series:
        notes.append(f"Lệch số chuỗi: series_name có {num_series} chuỗi nhưng values có {num_val_series} chuỗi")
        status = "Cần kiểm tra"

    # Đối chiếu số lượng giá trị trong từng chuỗi với số categories
    if num_cats > 0 and num_val_series > 0:
        for s_idx, val_str in enumerate(val_series_list):
            vals = [x.strip() for x in val_str.split("|") if x.strip()]
            if len(vals) != num_cats:
                s_name = series_list[s_idx] if s_idx < num_series else f"Chuỗi_{s_idx+1}"
                notes.append(f"Chuỗi '{s_name}' có {len(vals)} giá trị != {num_cats} categories")
                status = "Cần kiểm tra"

    # 5. Kiểm tra logic cơ cấu cho biểu đồ tròn hoặc miền (nếu unit có %)
    if ("tròn" in chart_type or "miền" in chart_type) and ("%" in unit or "%" in values_str):
        for s_idx, val_str in enumerate(val_series_list):
            vals = [x.strip() for x in val_str.split("|") if x.strip()]
            num_parsed = []
            for v in vals:
                # Chuẩn hóa tạm thời để tính tổng: đổi dấu phẩy thành dấu chấm
                v_clean = v.replace("%", "").replace(" ", "").replace(".", "").replace(",", ".")
                try:
                    num_parsed.append(float(v_clean))
                except ValueError:
                    pass
            if num_parsed and len(num_parsed) == len(vals):
                total_pct = sum(num_parsed)
                if 95.0 <= total_pct <= 105.0:
                    notes.append(f"Cơ cấu chuẩn: Tổng = {total_pct:.1f}%")
                else:
                    notes.append(f"Cảnh báo cơ cấu: Tổng = {total_pct:.1f}% (lệch nhiều so với 100%)")
                    status = "Cần kiểm tra"

    final_note = "; ".join(notes) if notes else "Đạt chuẩn định dạng & khớp số liệu"

    cleaned_dict = {
        "chart_title": chart_title,
        "chart_type": chart_type,
        "unit": unit,
        "x_axis_label": x_axis_label,
        "categories": categories_str,
        "series_name": series_name_str,
        "values": values_str
    }

    return status, final_note, cleaned_dict

print("✅ Khung kiểm chứng số liệu (Bước 2) đã sẵn sàng!")

--- 
## 🔄 Phần 6: Vòng Lặp Duyệt Ảnh, Trích Xuất & Ghi Kết Quả Cuốn Chiếu (Streaming Append)

**Các cơ chế bảo vệ phiên làm việc trên Colab:**
- **Resume Checkpoint:** Kiểm tra các ảnh đã trích xuất trong `OUTPUT_CSV`. Nếu tiến trình bị đứt kết nối giữa chừng, lần chạy tiếp theo sẽ tự động nhận diện và bỏ qua những ảnh đã có để chạy tiếp.
- **Append từng dòng ngay lập tức:** Dữ liệu được ghi thẳng xuống ổ đĩa Google Drive sau mỗi ảnh (`f.flush()`, `os.fsync()`) -> Không bao giờ lo mất dữ liệu nếu đứt runtime.
- **Chống tràn VRAM GPU:** Gọi `torch.cuda.empty_cache()` sau mỗi ảnh để bộ nhớ GPU T4 luôn ở mức an toàn (~4-5 GB VRAM).

In [ ]:
import os
import csv
import unicodedata
import pandas as pd
from tqdm import tqdm

# Định nghĩa cấu trúc cột của file CSV đầu ra (đúng 7 cột ground-truth + các cột kiểm soát)
CSV_FIELDNAMES = [
    "STT",
    "image_name",
    "chart_title",
    "chart_type",
    "unit",
    "x_axis_label",
    "categories",
    "series_name",
    "values",
    "validation_status",
    "validation_notes"
]

def natural_sort_key(s):
    """Sắp xếp tên file tự nhiên theo thứ tự số (ví dụ: bieu_do_2 đứng trước bieu_do_10)"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

# 1. Quét tìm toàn bộ file ảnh trong thư mục Drive (hỗ trợ cả các thư mục con)
print(f"🔍 Đang quét toàn bộ file ảnh trong: {IMAGE_DIR} ...")
image_dict = {}

if os.path.exists(IMAGE_DIR):
    for root, _, files in os.walk(IMAGE_DIR):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                norm_name = unicodedata.normalize("NFC", f).strip()
                image_dict[norm_name] = os.path.join(root, f)

sorted_image_names = sorted(list(image_dict.keys()), key=natural_sort_key)
print(f"-> Đã lập chỉ mục {len(sorted_image_names)} file ảnh sẵn sàng trích xuất.")

# 2. Kiểm tra Checkpoint (Resume) từ file kết quả cũ
processed_images = set()
if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0:
    try:
        df_old = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
        if "image_name" in df_old.columns:
            processed_images = set(df_old["image_name"].dropna().astype(str).str.strip())
            print(f"🔄 Checkpoint phát hiện {len(processed_images)} ảnh đã trích xuất trước đó. Tiếp tục với các ảnh còn lại!")
    except Exception as e:
        print(f"⚠️ Không đọc được file cũ ({e}), sẽ tạo mới.")

# 3. Khởi tạo file CSV nếu chưa tồn tại (Drive & Local backup)
if not os.path.exists(OUTPUT_CSV) or os.path.getsize(OUTPUT_CSV) == 0:
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    with open(OUTPUT_CSV, mode="w", encoding="utf-8-sig", newline="") as f_init:
        writer = csv.DictWriter(f_init, fieldnames=CSV_FIELDNAMES)
        writer.writeheader()
    print(f"✨ Đã tạo mới file kết quả: {OUTPUT_CSV}")

if not os.path.exists(LOCAL_CSV_BACKUP) or os.path.getsize(LOCAL_CSV_BACKUP) == 0:
    with open(LOCAL_CSV_BACKUP, mode="w", encoding="utf-8-sig", newline="") as f_local:
        writer = csv.DictWriter(f_local, fieldnames=CSV_FIELDNAMES)
        writer.writeheader()

# 4. Lập danh sách ảnh cần xử lý
images_to_process = [name for name in sorted_image_names if name not in processed_images]
print(f"🚀 Bắt đầu trích xuất {len(images_to_process)} ảnh...\n")

success_count = 0
warning_count = 0
error_count = 0
current_stt = len(processed_images)

# Duyệt qua từng ảnh với thanh tiến trình trực quan
for idx, img_name in enumerate(tqdm(images_to_process, desc="Tiến độ"), start=1):
    current_stt += 1
    img_path = image_dict[img_name]

    try:
        # Bước 1: Trích xuất số liệu bằng VLM
        data, raw_text, error_msg = extract_chart_data(img_path)

        if error_msg:
            status = "Lỗi trích xuất"
            notes = error_msg
            cleaned_data = {
                "chart_title": "", "chart_type": "", "unit": "",
                "x_axis_label": "", "categories": "", "series_name": "", "values": ""
            }
            error_count += 1
        else:
            # Bước 2: Kiểm chứng số liệu theo hệ luật
            status, notes, cleaned_data = validate_extracted_data(data)
            if status == "Hợp lệ":
                success_count += 1
            else:
                warning_count += 1

    except Exception as exc:
        status = "Lỗi runtime"
        notes = f"Exception: {str(exc)}"
        cleaned_data = {
            "chart_title": "", "chart_type": "", "unit": "",
            "x_axis_label": "", "categories": "", "series_name": "", "values": ""
        }
        error_count += 1

    # Chuẩn bị bản ghi kết quả
    row_record = {
        "STT": current_stt,
        "image_name": img_name,
        "chart_title": cleaned_data.get("chart_title", ""),
        "chart_type": cleaned_data.get("chart_type", ""),
        "unit": cleaned_data.get("unit", ""),
        "x_axis_label": cleaned_data.get("x_axis_label", ""),
        "categories": cleaned_data.get("categories", ""),
        "series_name": cleaned_data.get("series_name", ""),
        "values": cleaned_data.get("values", ""),
        "validation_status": status,
        "validation_notes": notes
    }

    # GHI CUỐN CHIẾU (APPEND) NGAY VÀO FILE DRIVE
    with open(OUTPUT_CSV, mode="a", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.DictWriter(f_out, fieldnames=CSV_FIELDNAMES)
        writer.writerow(row_record)
        f_out.flush()
        os.fsync(f_out.fileno())

    # Đồng bộ sang file local backup
    try:
        with open(LOCAL_CSV_BACKUP, mode="a", encoding="utf-8-sig", newline="") as f_loc:
            writer_loc = csv.DictWriter(f_loc, fieldnames=CSV_FIELDNAMES)
            writer_loc.writerow(row_record)
            f_loc.flush()
    except Exception:
        pass

    # In log trạng thái vắn tắt
    icon = "✅" if status == "Hợp lệ" else ("⚠️" if status == "Cần kiểm tra" else "❌")
    tqdm.write(f"{icon} [{current_stt}] {img_name} | {cleaned_data.get('chart_type')} | {status} ({notes[:50]})")

    # Giải phóng VRAM GPU sau mỗi ảnh
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "="*50)
print("🎉 ĐÃ HOÀN THÀNH ĐỢT TRÍCH XUẤT!")
print(f"- Đã xử lý đợt này: {len(images_to_process)} ảnh")
print(f"- Hợp lệ: {success_count} | Cần kiểm tra: {warning_count} | Lỗi: {error_count}")
print(f"- File CSV kết quả đã lưu an toàn tại: {OUTPUT_CSV}")
print("="*50)

--- 
## 📈 Phần 7: Thống kê Kết quả & Tải File CSV Đối chứng (Baseline)

Khối code dưới đây giúp nhóm:
1. Tổng hợp phân bố các loại biểu đồ (đối chiếu với tỉ lệ mục tiêu Slide 8: Cột 35%, Đường 25%, Tròn 20%, Miền 10%, Kết hợp 10%).
2. Thống kê tỉ lệ các mẫu đạt chuẩn `Hợp lệ` vs `Cần kiểm tra` để làm thước đo thực nghiệm ban đầu (Mức 1 theo Slide 9).
3. Xuất và tải trực tiếp file CSV về máy tính để import vào Google Sheets chuẩn của nhóm.

In [ ]:
import pandas as pd
from google.colab import files

# Kiểm tra và hiển thị thống kê từ file CSV
target_file = OUTPUT_CSV if (os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0) else LOCAL_CSV_BACKUP

if os.path.exists(target_file) and os.path.getsize(target_file) > 0:
    df_final = pd.read_csv(target_file, encoding="utf-8-sig")
    print(f"📊 BÁO CÁO TỔNG QUAN KẾT QUẢ ({len(df_final)} ảnh đã xử lý):\n" + "-"*50)

    # 1. Thống kê theo trạng thái kiểm chứng Bước 2
    print("\n1. Phân bố trạng thái kiểm chứng (Validation Status):")
    print(df_final["validation_status"].value_counts())

    # 2. Thống kê theo loại biểu đồ (chart_type)
    print("\n2. Phân bố loại biểu đồ nhận diện được:")
    print(df_final["chart_type"].value_counts())

    # 3. Xem 5 mẫu trích xuất thành công tiêu biểu
    print("\n3. Mẫu 5 dòng dữ liệu Hợp lệ:")
    valid_rows = df_final[df_final["validation_status"] == "Hợp lệ"]
    if not valid_rows.empty:
        display(valid_rows[["image_name", "chart_title", "chart_type", "unit", "categories", "series_name", "values"]].head(5))

    # 4. Xem các mẫu bị cảnh báo để nhóm ưu tiên rà soát thủ công
    warning_rows = df_final[df_final["validation_status"] != "Hợp lệ"]
    if not warning_rows.empty:
        print(f"\n4. Mẫu {min(5, len(warning_rows))} trường hợp cần kiểm tra lại:")
        display(warning_rows[["image_name", "chart_type", "validation_status", "validation_notes"]].head(5))

    # 5. Kích hoạt tải file CSV về máy tính
    print("\n📥 Đang kích hoạt tải file CSV về máy tính...")
    try:
        files.download(target_file)
        print("✅ Trình duyệt đang tải file xuống máy tính của bạn.")
    except Exception as e:
        print(f"ℹ️ Nếu không thấy popup tải file, bạn có thể tải thủ công tại đường dẫn: {target_file}")
else:
    print(f"⚠️ Chưa tìm thấy file dữ liệu tại {target_file}.")